# Ejemplo de encodec arq.

## Deepseek

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualUnit(nn.Module):
    def __init__(self, channels):
        super(ResidualUnit, self).__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, padding=1)
        self.activation = nn.ELU()
        self.norm = nn.LayerNorm(channels)  # Normalizes over the last dimension

    def forward(self, x):
        residual = x
        x = self.conv1(x)
        x = self.activation(x)
        
        # Transpose for LayerNorm: [batch, channels, time] -> [batch, time, channels]
        x = x.permute(0, 2, 1)
        x = self.norm(x)
        # Transpose back: [batch, time, channels] -> [batch, channels, time]
        x = x.permute(0, 2, 1)
        
        x = self.conv2(x)
        x += residual
        x = self.activation(x)
        return x

class Encoder(nn.Module):
    def __init__(self, C=32, B=4, strides=[2, 4, 5, 8], D=32):
        super(Encoder, self).__init__()
        self.initial_conv = nn.Conv1d(1, C, kernel_size=7, padding=3)
        self.blocks = nn.ModuleList()
        channels = C
        for stride in strides:
            self.blocks.append(ResidualUnit(channels))
            self.blocks.append(nn.Conv1d(channels, channels * 2, kernel_size=2 * stride, stride=stride, padding=stride))
            channels *= 2
        self.lstm = nn.LSTM(channels, channels, num_layers=2, batch_first=True)
        self.final_conv = nn.Conv1d(channels, D, kernel_size=7, padding=3)

    def forward(self, x):
        x = self.initial_conv(x)
        for block in self.blocks:
            x = block(x)
        x = x.permute(0, 2, 1)  # Prepare for LSTM
        x, _ = self.lstm(x)
        x = x.permute(0, 2, 1)  # Prepare for final conv
        x = self.final_conv(x)
        return x

# Decoder would be similar but in reverse, with upsampling instead of downsampling

In [8]:
# Parameters
sample_rate = 24000
duration = 2  # seconds
signal_length = sample_rate * duration

# Create a synthetic signal (e.g., a sine wave)
t = torch.linspace(0, duration, signal_length)
signal = torch.sin(2 * torch.pi * 440 * t)  # 440 Hz sine wave

# Add batch and channel dimensions
signal = signal.unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, 48000)

# Initialize the encoder
encoder = Encoder()

# Process the signal through the encoder
encoded_signal = encoder(signal)

print(f"Encoded signal shape: {encoded_signal.shape}")

Encoded signal shape: torch.Size([1, 32, 151])


In [9]:
from torchviz import make_dot

# Generate a visualization of the encoder
make_dot(encoded_signal, params=dict(encoder.named_parameters())).render("encoder_architecture", format="png")

CalledProcessError: Command '[PosixPath('dot'), '-Kdot', '-Tpng', '-O', 'encoder_architecture']' died with <Signals.SIGSEGV: 11>. [stderr: b'']

# Ejemplo de Encodec

In [1]:
from datasets import load_dataset, Audio
from transformers import EncodecModel, AutoProcessor


In [3]:
librispeech_dummy = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/9.19M [00:00<?, ?B/s]

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

In [13]:
model = EncodecModel.from_pretrained("facebook/encodec_24khz")

config.json:   0%|          | 0.00/809 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/93.1M [00:00<?, ?B/s]

/Users/elece/Library/CloudStorage/GoogleDrive-ekinig@gmail.com/My Drive/Tesis-grado/v2/proyecto/.venv/lib/python3.10/site-packages/transformers/models/encodec/modeling_encodec.py:124: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.register_buffer("padding_total", torch.tensor(kernel_size - stride, dtype=torch.int64), persistent=False)


In [ ]:

librispeech_dummy = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")

model = EncodecModel.from_pretrained("facebook/encodec_24khz")
processor = AutoProcessor.from_pretrained("facebook/encodec_24khz")
librispeech_dummy = librispeech_dummy.cast_column("audio", Audio(sampling_rate=processor.sampling_rate))
audio_sample = librispeech_dummy[-1]["audio"]["array"]
inputs = processor(raw_audio=audio_sample, sampling_rate=processor.sampling_rate, return_tensors="pt")

encoder_outputs = model.encode(inputs["input_values"], inputs["padding_mask"])
audio_values = model.decode(encoder_outputs.audio_codes, encoder_outputs.audio_scales, inputs["padding_mask"])[0]
# or the equivalent with a forward pass
audio_values = model(inputs["input_values"], inputs["padding_mask"]).audio_values

In [2]:
# from encodec import EncodecModel
import librosa
import torch

def tokenize(filename, model, sr=24000, num_quantizers=8):
    with torch.no_grad():
        x, fs = librosa.core.load(filename, sr=sr)
        codes = model.encode(torch.from_numpy(x)[None,None,:])[0][0]
    return codes

def detokenize(codes, model):
    with torch.no_grad():
        decoded_x = model.decode([(codes,None)])
    return decoded_x.detach().cpu().numpy()[0,0]

NUM_QUANTIZERS = 8

model = EncodecModel.encodec_model_24khz()
codes = tokenize('/mnt/hdd6T/jamendo/00/565800.mp3', model)
reconstruction_8q = detokenize(codes[:,:NUM_QUANTIZERS], model)

AttributeError: type object 'EncodecModel' has no attribute 'encodec_model_24khz'